## Game Theory Pricing Simulator

In [21]:
"""
Game Theory Pricing Simulator - HTML/JavaScript Version
Runs entirely in your browser, no Jupyter widget issues
"""

from IPython.display import display, HTML

html_code = """
<!DOCTYPE html>
<html>
<head>
    <style>
        body {
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Arial, sans-serif;
            max-width: 1400px;
            margin: 0 auto;
            padding: 20px;
            background: #f5f7fa;
        }
        .header {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            padding: 20px;
            border-radius: 10px;
            color: white;
            margin-bottom: 20px;
            text-align: center;
        }
        .controls {
            background: white;
            padding: 20px;
            border-radius: 10px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        .control-group {
            display: inline-block;
            margin: 10px 20px;
            min-width: 200px;
        }
        .control-group label {
            display: block;
            font-weight: bold;
            margin-bottom: 5px;
            color: #333;
        }
        .control-group input {
            width: 180px;
            padding: 8px;
            border: 1px solid #ddd;
            border-radius: 5px;
            font-size: 14px;
        }
        button {
            background: #28a745;
            color: white;
            border: none;
            padding: 12px 30px;
            font-size: 16px;
            border-radius: 5px;
            cursor: pointer;
            margin: 20px 0;
            font-weight: bold;
        }
        button:hover {
            background: #218838;
        }
        .results {
            background: white;
            padding: 20px;
            border-radius: 10px;
            margin-top: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        .result-card {
            background: #f8f9fa;
            padding: 15px;
            margin: 10px 0;
            border-radius: 5px;
            border-left: 4px solid #667eea;
        }
        .profit {
            font-size: 24px;
            font-weight: bold;
            color: #28a745;
        }
        .warning {
            color: #dc3545;
        }
        .success {
            color: #28a745;
        }
        .chart-container {
            margin: 20px 0;
            text-align: center;
        }
        canvas {
            max-width: 100%;
            background: white;
            border-radius: 5px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        hr {
            margin: 20px 0;
        }
        h3 {
            color: #333;
            margin-top: 20px;
        }
        .definition {
            background: #e8f4f8;
            padding: 15px;
            border-radius: 5px;
            margin-top: 20px;
            font-size: 14px;
        }
    </style>
</head>
<body>
    <div class="header">
        <h1>🎮 Game Theory Pricing Simulator</h1>
        <p>Adjust values below, then click "Calculate" to see results</p>
    </div>
    
    <div class="controls">
        <div class="control-group">
            <label>Market Size:</label>
            <input type="number" id="marketSize" value="10000" step="1000">
        </div>
        <div class="control-group">
            <label>Price Sensitivity:</label>
            <input type="number" id="priceSensitivity" value="2.5" step="0.1" min="1" max="5">
        </div>
        <div class="control-group">
            <label>Switching Cost ($):</label>
            <input type="number" id="switchingCost" value="50" step="10">
        </div>
        <div class="control-group">
            <label>Brand Strength:</label>
            <input type="number" id="brandStrength" value="0.6" step="0.05" min="0" max="1">
        </div>
        <div class="control-group">
            <label>Your Price ($):</label>
            <input type="number" id="priceA" value="150" step="5">
        </div>
        <div class="control-group">
            <label>Competitor Price ($):</label>
            <input type="number" id="priceB" value="150" step="5">
        </div>
        <div style="text-align: center;">
            <button onclick="calculate()">▶ CALCULATE / UPDATE</button>
        </div>
    </div>
    
    <div id="results" class="results">
        <div style="text-align: center; color: #666;">Click "Calculate" to see results</div>
    </div>
</body>

<script>
function calculate() {
    // Get values
    const marketSize = parseFloat(document.getElementById('marketSize').value);
    const priceSensitivity = parseFloat(document.getElementById('priceSensitivity').value);
    const switchingCost = parseFloat(document.getElementById('switchingCost').value);
    const brandStrength = parseFloat(document.getElementById('brandStrength').value);
    const priceA = parseFloat(document.getElementById('priceA').value);
    const priceB = parseFloat(document.getElementById('priceB').value);
    
    const costA = 50;
    const costB = 55;
    
    // Calculate market share (logit model)
    const brandEffect = brandStrength - 0.5;
    const switchingEffect = switchingCost / 100;
    
    const utilityA = -priceSensitivity * priceA + brandEffect + switchingEffect;
    const utilityB = -priceSensitivity * priceB;
    
    const shareA = 1 / (1 + Math.exp(utilityB - utilityA));
    shareA_clipped = Math.min(0.95, Math.max(0.05, shareA));
    const shareB = 1 - shareA_clipped;
    
    // Calculate demand and profit
    const demandA = marketSize * shareA_clipped;
    const demandB = marketSize * shareB;
    
    const profitA = (priceA - costA) * demandA;
    const profitB = (priceB - costB) * demandB;
    
    // Find Nash equilibrium (simplified)
    function bestResponseA(priceB) {
        let bestPrice = priceA;
        let bestProfit = profitA;
        for (let p = 80; p <= 300; p += 5) {
            const brandEffect2 = brandStrength - 0.5;
            const switchingEffect2 = switchingCost / 100;
            const utilA = -priceSensitivity * p + brandEffect2 + switchingEffect2;
            const utilB = -priceSensitivity * priceB;
            const share = 1 / (1 + Math.exp(utilB - utilA));
            const clippedShare = Math.min(0.95, Math.max(0.05, share));
            const demand = marketSize * clippedShare;
            const profit = (p - costA) * demand;
            if (profit > bestProfit) {
                bestProfit = profit;
                bestPrice = p;
            }
        }
        return bestPrice;
    }
    
    function bestResponseB(priceA) {
        let bestPrice = priceB;
        let bestProfit = profitB;
        for (let p = 80; p <= 300; p += 5) {
            const brandEffect2 = brandStrength - 0.5;
            const switchingEffect2 = switchingCost / 100;
            const utilA = -priceSensitivity * priceA + brandEffect2 + switchingEffect2;
            const utilB = -priceSensitivity * p;
            const shareA = 1 / (1 + Math.exp(utilB - utilA));
            const shareB = 1 - shareA;
            const demand = marketSize * shareB;
            const profit = (p - costB) * demand;
            if (profit > bestProfit) {
                bestProfit = profit;
                bestPrice = p;
            }
        }
        return bestPrice;
    }
    
    let nashPriceA = 150;
    let nashPriceB = 150;
    for (let i = 0; i < 20; i++) {
        const newA = bestResponseA(nashPriceB);
        const newB = bestResponseB(newA);
        if (Math.abs(newA - nashPriceA) < 1 && Math.abs(newB - nashPriceB) < 1) break;
        nashPriceA = newA;
        nashPriceB = newB;
    }
    
    const nashProfitA = (nashPriceA - costA) * marketSize * 
        (1 / (1 + Math.exp((-priceSensitivity * nashPriceB) - (-priceSensitivity * nashPriceA + (brandStrength-0.5) + switchingCost/100))));
    
    // Calculate elasticity
    const epsilon = 5;
    const shareHigh = 1 / (1 + Math.exp((-priceSensitivity * priceB) - (-priceSensitivity * (priceA - epsilon) + (brandStrength-0.5) + switchingCost/100)));
    const shareLow = 1 / (1 + Math.exp((-priceSensitivity * priceB) - (-priceSensitivity * (priceA + epsilon) + (brandStrength-0.5) + switchingCost/100)));
    const pctChangeDemand = (shareLow - shareHigh) / shareHigh;
    const pctChangePrice = (2 * epsilon) / priceA;
    const elasticity = (pctChangeDemand / pctChangePrice) * -1;
    
    // Prepare profit curve data
    const prices = [];
    const profits = [];
    for (let p = 80; p <= 300; p += 5) {
        const brandEffect2 = brandStrength - 0.5;
        const switchingEffect2 = switchingCost / 100;
        const utilA = -priceSensitivity * p + brandEffect2 + switchingEffect2;
        const utilB = -priceSensitivity * priceB;
        const share = 1 / (1 + Math.exp(utilB - utilA));
        const clippedShare = Math.min(0.95, Math.max(0.05, share));
        const demand = marketSize * clippedShare;
        const profit = (p - costA) * demand;
        prices.push(p);
        profits.push(profit);
    }
    
    // Prepare market share data
    const shares = [];
    for (let p = 80; p <= 300; p += 5) {
        const brandEffect2 = brandStrength - 0.5;
        const switchingEffect2 = switchingCost / 100;
        const utilA = -priceSensitivity * p + brandEffect2 + switchingEffect2;
        const utilB = -priceSensitivity * priceB;
        const share = 1 / (1 + Math.exp(utilB - utilA));
        shares.push(share * 100);
    }
    
    // Display results
    const resultsDiv = document.getElementById('results');
    resultsDiv.innerHTML = `
        <div style="background: #f8f9fa; padding: 15px; border-radius: 5px; margin-bottom: 20px;">
            <h3 style="margin-top: 0;">📊 CURRENT RESULTS</h3>
            <div style="display: inline-block; width: 45%; margin: 10px;">
                <div style="background: white; padding: 15px; border-radius: 5px;">
                    <strong>💰 YOUR FIRM (Firm A)</strong><br>
                    Price: $${priceA}<br>
                    Profit: $${profitA.toLocaleString()}<br>
                    Market Share: ${(shareA_clipped*100).toFixed(1)}%<br>
                    Customers: ${Math.round(demandA).toLocaleString()}
                </div>
            </div>
            <div style="display: inline-block; width: 45%; margin: 10px;">
                <div style="background: white; padding: 15px; border-radius: 5px;">
                    <strong>🏢 COMPETITOR (Firm B)</strong><br>
                    Price: $${priceB}<br>
                    Profit: $${profitB.toLocaleString()}<br>
                    Market Share: ${(shareB*100).toFixed(1)}%<br>
                    Customers: ${Math.round(demandB).toLocaleString()}
                </div>
            </div>
        </div>
        
        <div class="result-card">
            <h3>🎯 NASH EQUILIBRIUM</h3>
            <p><strong>Your Optimal Price:</strong> $${nashPriceA.toFixed(2)}</p>
            <p><strong>Competitor Optimal Price:</strong> $${nashPriceB.toFixed(2)}</p>
            <p><strong>Your Profit at Nash:</strong> $${(nashProfitA).toLocaleString()}</p>
            <p><strong>Potential Gain:</strong> $${(nashProfitA - profitA).toLocaleString()}</p>
            ${nashPriceA > priceA ? '<p class="success">→ RECOMMENDATION: RAISE price to $' + nashPriceA.toFixed(2) + '</p>' : 
              nashPriceA < priceA ? '<p class="warning">→ RECOMMENDATION: LOWER price to $' + nashPriceA.toFixed(2) + '</p>' :
              '<p class="success">→ RECOMMENDATION: Current price is optimal</p>'}
        </div>
        
        <div class="result-card">
            <h3>📈 PRICE ELASTICITY</h3>
            <p><strong>Own Price Elasticity:</strong> ${elasticity.toFixed(3)}</p>
            ${elasticity > 1 ? 
                '<p>→ ELASTIC DEMAND: Customers ARE price-sensitive</p>' :
                '<p>→ INELASTIC DEMAND: Customers are NOT price-sensitive</p>'}
        </div>
        
        <div class="definition">
            <h3>📚 PARAMETER DEFINITIONS</h3>
            <p><strong>Market Size (${marketSize.toLocaleString()}):</strong> Total potential customers in the market</p>
            <p><strong>Price Sensitivity (${priceSensitivity}):</strong> How much customers react to price (1=low, 5=high)</p>
            <p><strong>Switching Cost ($${switchingCost}):</strong> Cost for customers to switch to competitor</p>
            <p><strong>Brand Strength (${brandStrength}):</strong> Your brand advantage (0=weak, 1=strong)</p>
            <p><strong>Your Price ($${priceA}):</strong> Your current price per unit</p>
            <p><strong>Competitor Price ($${priceB}):</strong> Competitor's current price</p>
        </div>
        
        <div class="chart-container">
            <h3>📊 Profit vs Price (Competitor Fixed)</h3>
            <canvas id="profitChart" width="600" height="300"></canvas>
        </div>
        
        <div class="chart-container">
            <h3>📊 Market Share vs Price</h3>
            <canvas id="shareChart" width="600" height="300"></canvas>
        </div>
    `;
    
    // Draw profit chart
    const profitCanvas = document.getElementById('profitChart');
    if (profitCanvas) {
        const ctx = profitCanvas.getContext('2d');
        const width = profitCanvas.width;
        const height = profitCanvas.height;
        
        ctx.clearRect(0, 0, width, height);
        
        // Draw axes
        ctx.beginPath();
        ctx.strokeStyle = '#333';
        ctx.lineWidth = 1;
        ctx.moveTo(50, 20);
        ctx.lineTo(50, height - 40);
        ctx.lineTo(width - 20, height - 40);
        ctx.stroke();
        
        // Find max profit for scaling
        const maxProfit = Math.max(...profits);
        const minProfit = Math.min(...profits);
        
        // Draw profit curve
        ctx.beginPath();
        ctx.strokeStyle = '#2E86AB';
        ctx.lineWidth = 2;
        
        for (let i = 0; i < prices.length; i++) {
            const x = 50 + (prices[i] - 80) / 220 * (width - 70);
            const y = height - 40 - ((profits[i] - minProfit) / (maxProfit - minProfit) * (height - 60));
            if (i === 0) ctx.moveTo(x, y);
            else ctx.lineTo(x, y);
        }
        ctx.stroke();
        
        // Draw current price line
        const currentX = 50 + (priceA - 80) / 220 * (width - 70);
        ctx.beginPath();
        ctx.strokeStyle = 'red';
        ctx.setLineDash([5, 5]);
        ctx.moveTo(currentX, 20);
        ctx.lineTo(currentX, height - 40);
        ctx.stroke();
        ctx.setLineDash([]);
        
        // Labels
        ctx.fillStyle = '#333';
        ctx.font = '12px Arial';
        ctx.fillText('Price ($)', width/2, height - 10);
        ctx.save();
        ctx.translate(20, height/2);
        ctx.rotate(-Math.PI/2);
        ctx.fillText('Profit ($)', 0, 0);
        ctx.restore();
        
        // Draw current profit point
        const currentProfitY = height - 40 - ((profitA - minProfit) / (maxProfit - minProfit) * (height - 60));
        ctx.beginPath();
        ctx.fillStyle = 'red';
        ctx.arc(currentX, currentProfitY, 5, 0, 2 * Math.PI);
        ctx.fill();
    }
    
    // Draw market share chart
    const shareCanvas = document.getElementById('shareChart');
    if (shareCanvas) {
        const ctx = shareCanvas.getContext('2d');
        const width = shareCanvas.width;
        const height = shareCanvas.height;
        
        ctx.clearRect(0, 0, width, height);
        
        // Draw axes
        ctx.beginPath();
        ctx.strokeStyle = '#333';
        ctx.moveTo(50, 20);
        ctx.lineTo(50, height - 40);
        ctx.lineTo(width - 20, height - 40);
        ctx.stroke();
        
        // Draw share curve
        ctx.beginPath();
        ctx.strokeStyle = '#27AE60';
        ctx.lineWidth = 2;
        
        for (let i = 0; i < prices.length; i++) {
            const x = 50 + (prices[i] - 80) / 220 * (width - 70);
            const y = height - 40 - (shares[i] / 100 * (height - 60));
            if (i === 0) ctx.moveTo(x, y);
            else ctx.lineTo(x, y);
        }
        ctx.stroke();
        
        // Draw current price line
        const currentX = 50 + (priceA - 80) / 220 * (width - 70);
        ctx.beginPath();
        ctx.strokeStyle = 'red';
        ctx.setLineDash([5, 5]);
        ctx.moveTo(currentX, 20);
        ctx.lineTo(currentX, height - 40);
        ctx.stroke();
        ctx.setLineDash([]);
        
        // Labels
        ctx.fillStyle = '#333';
        ctx.fillText('Price ($)', width/2, height - 10);
        ctx.save();
        ctx.translate(20, height/2);
        ctx.rotate(-Math.PI/2);
        ctx.fillText('Market Share (%)', 0, 0);
        ctx.restore();
        
        // Draw current share point
        const currentShareY = height - 40 - ((shareA_clipped * 100) / 100 * (height - 60));
        ctx.beginPath();
        ctx.fillStyle = 'red';
        ctx.arc(currentX, currentShareY, 5, 0, 2 * Math.PI);
        ctx.fill();
    }
}

// Run initial calculation
calculate();
</script>
</html>
"""

display(HTML(html_code))